# Create Hong Kong RGC Project Awards

Creates OpenAlex award rows from the Research Grants Council (RGC), University Grants Committee public project-enquiry portal.

**Prerequisites:**
- Admin/human with AWS credentials runs `scripts/local/rgc_hk_to_s3.py` to fetch the official RGC public enquiry pages, write parquet, and upload it.
- Contractor has prepared the script, notebook, registry entry, tracker row, and local QA, but does not have S3/Databricks access.
- The downloader writes parquet columns as strings per `plans/awards/how-to-add-a-funder.md`; this notebook does type conversion with `TRY_CAST` / `TRY_TO_DATE`.

**Data source:** Official RGC funded-research page links the public Project Enquiry portal at `https://cerg1.ugc.edu.hk/cergprod/scrrm00541.jsp`; detail pages are served by `scrrm00542.jsp`.

**S3 location:** `s3a://openalex-ingest/awards/rgc_hk/rgc_hk_projects.parquet`

**Local full-source validation on 2026-06-02:** 21,320 official RGC project rows, 2006-2025, 21,320 unique native project numbers, 100% title/PI/institution/panel/subject/amount/currency/start-year coverage, 76.2% abstract coverage, 99.4% completion-date coverage, HKD 14,815,985,901 total approved funding.

**OpenAlex funder mapping:** Research Grants Council, University Grants Committee `F4320321592` (HK, no ROR/DOI in the OpenAlex API response).

**Mapping summary:** One OpenAlex award row per native RGC project number. `Fund Approved` maps to amount in HKD. `Project Title(English)` maps to display name, `Abstract as per original application` maps to description when present, `Funding Scheme` maps to funder_scheme, and `Project Number` maps to funder_award_id. The PI maps to lead_investigator with institution as affiliation.name. The first source co-investigator maps to co_lead_investigator; remaining co-investigators map to investigators. Co-investigator affiliations and all country fields are NULL because RGC does not publish per-person affiliation/country fields for co-investigators or a per-project country column.

**Provenance:** `rgc_hk_project_enquiry`  
**Priority:** 198


## Step 1: Create Raw Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.rgc_hk_raw
USING delta
AS
SELECT
    *,
    current_timestamp() AS databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/rgc_hk/rgc_hk_projects.parquet`;

In [ ]:
%sql
-- Check raw row count. Local validation on 2026-06-02 found 21,320 rows.
SELECT COUNT(*) AS total_rgc_projects
FROM openalex.awards.rgc_hk_raw;

In [ ]:
%sql
-- Verify actual column names before transform logic references them.
DESCRIBE openalex.awards.rgc_hk_raw;

In [ ]:
%sql
-- Sample raw RGC data.
SELECT
    funder_award_id,
    funding_scheme,
    title,
    principal_investigator,
    institution,
    amount,
    currency,
    start_year,
    completion_date,
    project_status
FROM openalex.awards.rgc_hk_raw
ORDER BY start_year DESC, funder_award_id
LIMIT 20;

## Step 1.5: Source Data Scans

In [ ]:
%sql
-- Uniqueness and year range.
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT funder_award_id) AS distinct_project_numbers,
    MIN(TRY_CAST(start_year AS INT)) AS min_start_year,
    MAX(TRY_CAST(start_year AS INT)) AS max_start_year,
    MIN(TRY_CAST(end_year AS INT)) AS min_end_year,
    MAX(TRY_CAST(end_year AS INT)) AS max_end_year
FROM openalex.awards.rgc_hk_raw;

In [ ]:
%sql
-- Required source-field coverage.
SELECT
    COUNT(*) AS total_rows,
    COUNT(title) AS rows_with_title,
    COUNT(description) AS rows_with_description,
    COUNT(principal_investigator) AS rows_with_pi,
    COUNT(institution) AS rows_with_institution,
    COUNT(panel) AS rows_with_panel,
    COUNT(subject_area) AS rows_with_subject_area,
    COUNT(completion_date) AS rows_with_completion_date,
    ROUND(try_divide(COUNT(title), COUNT(*)) * 100.0, 1) AS pct_title,
    ROUND(try_divide(COUNT(description), COUNT(*)) * 100.0, 1) AS pct_description,
    ROUND(try_divide(COUNT(principal_investigator), COUNT(*)) * 100.0, 1) AS pct_pi,
    ROUND(try_divide(COUNT(institution), COUNT(*)) * 100.0, 1) AS pct_institution,
    ROUND(try_divide(COUNT(completion_date), COUNT(*)) * 100.0, 1) AS pct_completion_date
FROM openalex.awards.rgc_hk_raw;

In [ ]:
%sql
-- Amount/currency coverage. Section 6.7 is not waived for this source.
SELECT
    COUNT(*) AS total_rows,
    COUNT(amount) AS rows_with_amount,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) AS pct_amount,
    COUNT(currency) AS rows_with_currency,
    collect_set(currency) AS currencies,
    ROUND(SUM(TRY_CAST(amount AS DOUBLE)), 2) AS total_hkd_approved,
    ROUND(percentile_approx(TRY_CAST(amount AS DOUBLE), 0.5), 2) AS median_hkd_approved,
    ROUND(MAX(TRY_CAST(amount AS DOUBLE)), 2) AS max_hkd_approved
FROM openalex.awards.rgc_hk_raw;

In [ ]:
%sql
-- Scheme, status, panel, and institution distribution.
SELECT 'funding_scheme' AS dimension, funding_scheme AS value, COUNT(*) AS rows
FROM openalex.awards.rgc_hk_raw
GROUP BY funding_scheme
UNION ALL
SELECT 'project_status' AS dimension, project_status AS value, COUNT(*) AS rows
FROM openalex.awards.rgc_hk_raw
GROUP BY project_status
UNION ALL
SELECT 'panel' AS dimension, panel AS value, COUNT(*) AS rows
FROM openalex.awards.rgc_hk_raw
GROUP BY panel
UNION ALL
SELECT 'institution' AS dimension, institution AS value, COUNT(*) AS rows
FROM openalex.awards.rgc_hk_raw
GROUP BY institution
ORDER BY dimension, rows DESC;

In [ ]:
%sql
-- Co-investigator coverage.
SELECT
    COUNT(*) AS total_rows,
    COUNT(co_investigators_json) AS rows_with_co_investigators,
    ROUND(try_divide(COUNT(co_investigators_json), COUNT(*)) * 100.0, 1) AS pct_with_co_investigators
FROM openalex.awards.rgc_hk_raw;

## Step 1.6: Funder Existence Fail-Fast

The transform in Step 2 does `CROSS JOIN openalex.common.funder WHERE funder_id = 4320321592`. If this returns zero rows, STOP; the insert would silently emit no awards.


In [ ]:
%sql
-- Fails the cell if the Crossref-registered funder row is missing from openalex.common.funder.
SELECT CASE
    WHEN COUNT(*) = 1 THEN 'OK: RGC funder row exists'
    ELSE raise_error('RGC funder row missing from openalex.common.funder')
END AS funder_check
FROM openalex.common.funder
WHERE funder_id = 4320321592;

In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi
FROM openalex.common.funder
WHERE funder_id = 4320321592;

## Step 2: Transform to OpenAlex Awards Schema

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.rgc_hk_awards
USING delta
AS
WITH
rgc_funder AS (
    SELECT
        funder_id,
        display_name,
        ror_id,
        doi
    FROM openalex.common.funder
    WHERE funder_id = 4320321592
),

raw_prepared AS (
    SELECT
        *,
        LOWER(TRIM(funder_award_id)) AS native_award_id,
        TRY_CAST(amount AS DOUBLE) AS parsed_amount,
        TRY_CAST(start_year AS INT) AS parsed_start_year,
        TRY_CAST(end_year AS INT) AS parsed_end_year,
        TRY_CAST(completion_date AS DATE) AS parsed_completion_date,
        from_json(
            co_investigators_struct_json,
            'ARRAY<STRUCT<raw_name:STRING,given_name:STRING,family_name:STRING>>'
        ) AS co_people
    FROM openalex.awards.rgc_hk_raw
    WHERE funder_award_id IS NOT NULL
      AND TRIM(funder_award_id) != ''
      AND title IS NOT NULL
      AND TRIM(title) != ''
),

awards_transformed AS (
    SELECT
        abs(xxhash64(CONCAT(CAST(f.funder_id AS STRING), ':', g.native_award_id))) % 9000000000 AS id,
        TRIM(g.title) AS display_name,
        NULLIF(TRIM(g.description), '') AS description,
        f.funder_id,
        g.native_award_id AS funder_award_id,
        CASE WHEN g.parsed_amount > 0 THEN g.parsed_amount ELSE NULL END AS amount,
        CASE WHEN g.parsed_amount > 0 THEN 'HKD' ELSE NULL END AS currency,
        struct(
            CONCAT('https://openalex.org/F', f.funder_id) AS id,
            f.display_name,
            f.ror_id,
            f.doi
        ) AS funder,
        COALESCE(NULLIF(TRIM(g.funding_type_hint), ''), 'research') AS funding_type,
        NULLIF(TRIM(g.funding_scheme), '') AS funder_scheme,
        'rgc_hk_project_enquiry' AS provenance,
        CAST(NULL AS DATE) AS start_date,
        g.parsed_completion_date AS end_date,
        g.parsed_start_year AS start_year,
        g.parsed_end_year AS end_year,
        struct(
            NULLIF(TRIM(g.pi_given_name), '') AS given_name,
            NULLIF(TRIM(g.pi_family_name), '') AS family_name,
            CAST(NULL AS STRING) AS orcid,
            CAST(NULL AS DATE) AS role_start,
            struct(
                NULLIF(TRIM(g.institution), '') AS name,
                CAST(NULL AS STRING) AS country,
                CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) AS ids
            ) AS affiliation
        ) AS lead_investigator,
        CASE
            WHEN g.co_people IS NOT NULL AND size(g.co_people) > 0 THEN struct(
                NULLIF(TRIM(g.co_people[0].given_name), '') AS given_name,
                NULLIF(TRIM(g.co_people[0].family_name), '') AS family_name,
                CAST(NULL AS STRING) AS orcid,
                CAST(NULL AS DATE) AS role_start,
                struct(
                    CAST(NULL AS STRING) AS name,
                    CAST(NULL AS STRING) AS country,
                    CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) AS ids
                ) AS affiliation
            )
            ELSE CAST(NULL AS STRUCT<
                given_name:STRING,
                family_name:STRING,
                orcid:STRING,
                role_start:DATE,
                affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
            >)
        END AS co_lead_investigator,
        CASE
            WHEN g.co_people IS NOT NULL AND size(g.co_people) > 1 THEN transform(
                slice(g.co_people, 2, size(g.co_people) - 1),
                x -> struct(
                    NULLIF(TRIM(x.given_name), '') AS given_name,
                    NULLIF(TRIM(x.family_name), '') AS family_name,
                    CAST(NULL AS STRING) AS orcid,
                    CAST(NULL AS DATE) AS role_start,
                    struct(
                        CAST(NULL AS STRING) AS name,
                        CAST(NULL AS STRING) AS country,
                        CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) AS ids
                    ) AS affiliation
                )
            )
            ELSE CAST(NULL AS ARRAY<STRUCT<
                given_name:STRING,
                family_name:STRING,
                orcid:STRING,
                role_start:DATE,
                affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
            >>)
        END AS investigators,
        NULLIF(TRIM(g.source_url), '') AS landing_page_url,
        CAST(NULL AS STRING) AS doi,
        CONCAT('https://api.openalex.org/works?filter=awards.id:G',
               CAST(abs(xxhash64(CONCAT(CAST(f.funder_id AS STRING), ':', g.native_award_id))) % 9000000000 AS STRING)) AS works_api_url,
        current_timestamp() AS created_date,
        current_timestamp() AS updated_date
    FROM raw_prepared g
    CROSS JOIN rgc_funder f
)
SELECT
    * EXCEPT(start_year, end_year),
    CASE WHEN start_year > YEAR(current_date()) + 1 THEN NULL ELSE start_year END AS start_year,
    CASE WHEN start_year > YEAR(current_date()) + 1 THEN NULL ELSE end_year END AS end_year
FROM awards_transformed;

In [ ]:
%sql
-- Confirm transformed row count and uniqueness before insert.
SELECT
    COUNT(*) AS total_awards,
    COUNT(DISTINCT funder_award_id) AS distinct_award_ids,
    MIN(start_year) AS min_start_year,
    MAX(start_year) AS max_start_year
FROM openalex.awards.rgc_hk_awards;

## Step 3: Delete Existing Priority Rows and Insert

In [ ]:
%sql
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'rgc_hk_project_enquiry' AND priority = 198;

INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id,
    display_name,
    description,
    funder_id,
    funder_award_id,
    amount,
    currency,
    funder,
    funding_type,
    funder_scheme,
    provenance,
    start_date,
    end_date,
    start_year,
    end_year,
    lead_investigator,
    co_lead_investigator,
    investigators,
    landing_page_url,
    doi,
    works_api_url,
    created_date,
    updated_date,
    198 AS priority
FROM openalex.awards.rgc_hk_awards;

## Step 6: Verification

In [ ]:
%sql
-- Verify inserted rows for this provenance/priority.
SELECT
    COUNT(*) AS inserted_rows,
    COUNT(DISTINCT funder_award_id) AS distinct_award_ids,
    MIN(start_year) AS min_start_year,
    MAX(start_year) AS max_start_year,
    COUNT(amount) AS rows_with_amount,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) AS pct_amount,
    ROUND(SUM(amount), 2) AS total_hkd_amount
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'rgc_hk_project_enquiry' AND priority = 198;

In [ ]:
%sql
-- Section 6.7 amount check: populated from official RGC Fund Approved field.
SELECT
    funding_type,
    funder_scheme,
    COUNT(*) AS rows,
    COUNT(amount) AS rows_with_amount,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) AS pct_amount,
    ROUND(SUM(amount), 2) AS total_hkd_amount
FROM openalex.awards.rgc_hk_awards
GROUP BY funding_type, funder_scheme
ORDER BY rows DESC;

In [ ]:
%sql
-- Investigator coverage after transform.
SELECT
    COUNT(*) AS total_rows,
    COUNT(lead_investigator.family_name) AS rows_with_pi_family,
    COUNT(lead_investigator.affiliation.name) AS rows_with_pi_affiliation,
    COUNT(co_lead_investigator.family_name) AS rows_with_co_lead,
    ROUND(try_divide(COUNT(lead_investigator.family_name), COUNT(*)) * 100.0, 1) AS pct_pi_family,
    ROUND(try_divide(COUNT(co_lead_investigator.family_name), COUNT(*)) * 100.0, 1) AS pct_co_lead
FROM openalex.awards.rgc_hk_awards;

In [ ]:
%sql
-- Duplicate ID guard.
SELECT funder_award_id, COUNT(*) AS rows
FROM openalex.awards.rgc_hk_awards
GROUP BY funder_award_id
HAVING COUNT(*) > 1
ORDER BY rows DESC
LIMIT 20;

In [ ]:
%sql
-- Spot-check a few transformed rows.
SELECT
    funder_award_id,
    display_name,
    amount,
    currency,
    funding_type,
    funder_scheme,
    start_year,
    end_date,
    lead_investigator,
    co_lead_investigator,
    landing_page_url
FROM openalex.awards.rgc_hk_awards
ORDER BY start_year DESC, funder_award_id
LIMIT 20;

## Handoff / Admin Notes

Contractor-complete handoff only. Contractor has no S3/Databricks access.

Admin must:
- Run `scripts/local/rgc_hk_to_s3.py` without `--skip-upload` to upload `s3://openalex-ingest/awards/rgc_hk/rgc_hk_projects.parquet`.
- Run this Databricks notebook and inspect Step 6 verification cells.
- QA the inserted rows before marking the tracker Complete and deciding whether/when to add scheduled job YAML.
